In [1]:
print('hello world')

hello world


In [2]:
import pandas as pd
import random
import torch.nn as nn
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
import threading

In [3]:
# Generate dataset
airlines = ["Indigo", "Air India", "SpiceJet", "Vistara"]
airports = ["Delhi", "Mumbai", "Bangalore", "Chennai", "Kolkata"]
data = []
for _ in range(500):
    airline = random.choice(airlines)
    source = random.choice(airports)
    destination = random.choice(airports)
    while source == destination:
        destination = random.choice(airports)
    distance = random.randint(300, 2500)
    departure_hour = random.randint(0, 23)
    weather = random.choice(["Clear", "Rain", "Fog"])
    if weather in ["Rain", "Fog"] or departure_hour > 20:
        delay = 1
    else:
        delay = 0
    data.append([airline, source, destination, distance, departure_hour, weather, delay])

df = pd.DataFrame(data, columns=[
    "airline", "source", "destination", "distance",
    "departure_hour", "weather", "delay"
])

df.to_csv("flights.csv", index=False)
print("Dataset created ")

# Load and preprocess data
df = pd.read_csv("flights.csv")
X = df.drop("delay", axis=1)
y = df["delay"]

encoders = {}
for col in ["airline", "source", "destination", "weather"]:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

# Train ML model
model_ml = RandomForestClassifier()
model_ml.fit(X_train.numpy(), y_train.numpy())
accuracy_ml = model_ml.score(X_test.numpy(), y_test.numpy())
print("ML Accuracy:", accuracy_ml)

# Define Neural Network
class FlightModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 32)
        self.fc2 = nn.Linear(32, 16)
        self.out = nn.Linear(16, 2)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.out(x)

# Train DL model
model_dl = FlightModel(X_train.shape[1])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_dl.parameters(), lr=0.001)

for epoch in range(50):
    outputs = model_dl(X_train)
    loss = criterion(outputs, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

with torch.no_grad():
    outputs = model_dl(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy_dl = (predicted == y_test).sum().item() / len(y_test)
print("DL Accuracy:", accuracy_dl)

# Prediction function
def predict_flight(airline, source, destination, distance, hour, weather):
    data = pd.DataFrame([[airline, source, destination, distance, hour, weather]],
                        columns=X.columns)

    for col in ["airline", "source", "destination", "weather"]:
        data[col] = encoders[col].transform(data[col])

    data = scaler.transform(data)
    tensor = torch.tensor(data, dtype=torch.float32)

    with torch.no_grad():
        output = model_dl(tensor)
        _, pred = torch.max(output, 1)

    return "Delay" if pred.item() == 1 else "On Time"

# Tkinter GUI
class FlightPredictorApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Flight Delay Predictor")
        self.root.geometry("1200x800")
        
        # Title
        title_label = tk.Label(root, text="Flight Delay Prediction System", 
                              font=("Arial", 20, "bold"))
        title_label.pack(pady=10)
        
        # Accuracy info
        accuracy_frame = tk.Frame(root)
        accuracy_frame.pack(pady=5)
        tk.Label(accuracy_frame, text=f"ML Model Accuracy: {accuracy_ml:.3f}", 
                font=("Arial", 10)).pack(side=tk.LEFT, padx=10)
        tk.Label(accuracy_frame, text=f"DL Model Accuracy: {accuracy_dl:.3f}", 
                font=("Arial", 10)).pack(side=tk.LEFT, padx=10)
        
        # Main frame
        main_frame = tk.Frame(root)
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Left frame - Input and Predict
        left_frame = tk.Frame(main_frame)
        left_frame.pack(side=tk.LEFT, fill=tk.Y, padx=(0, 10))
        
        # Input frame
        input_frame = tk.LabelFrame(left_frame, text="Predict New Flight", font=("Arial", 12, "bold"))
        input_frame.pack(fill=tk.X, pady=(0, 10))
        
        # Input fields
        tk.Label(input_frame, text="Airline:").grid(row=0, column=0, sticky=tk.W, padx=5, pady=5)
        self.airline_var = tk.StringVar(value="Indigo")
        airline_combo = ttk.Combobox(input_frame, textvariable=self.airline_var, 
                                   values=airlines, state="readonly", width=15)
        airline_combo.grid(row=0, column=1, padx=5, pady=5)
        
        tk.Label(input_frame, text="Source:").grid(row=1, column=0, sticky=tk.W, padx=5, pady=5)
        self.source_var = tk.StringVar(value="Delhi")
        source_combo = ttk.Combobox(input_frame, textvariable=self.source_var, 
                                  values=airports, state="readonly", width=15)
        source_combo.grid(row=1, column=1, padx=5, pady=5)
        
        tk.Label(input_frame, text="Destination:").grid(row=2, column=0, sticky=tk.W, padx=5, pady=5)
        self.dest_var = tk.StringVar(value="Mumbai")
        dest_combo = ttk.Combobox(input_frame, textvariable=self.dest_var, 
                                values=airports, state="readonly", width=15)
        dest_combo.grid(row=2, column=1, padx=5, pady=5)
        
        tk.Label(input_frame, text="Distance (km):").grid(row=3, column=0, sticky=tk.W, padx=5, pady=5)
        self.distance_var = tk.StringVar(value="1400")
        tk.Entry(input_frame, textvariable=self.distance_var, width=17).grid(row=3, column=1, padx=5, pady=5)
        
        tk.Label(input_frame, text="Departure Hour:").grid(row=4, column=0, sticky=tk.W, padx=5, pady=5)
        self.hour_var = tk.StringVar(value="22")
        hour_combo = ttk.Combobox(input_frame, textvariable=self.hour_var, 
                                values=[str(i) for i in range(24)], state="readonly", width=15)
        hour_combo.grid(row=4, column=1, padx=5, pady=5)
        
        tk.Label(input_frame, text="Weather:").grid(row=5, column=0, sticky=tk.W, padx=5, pady=5)
        self.weather_var = tk.StringVar(value="Fog")
        weather_combo = ttk.Combobox(input_frame, textvariable=self.weather_var, 
                                   values=["Clear", "Rain", "Fog"], state="readonly", width=15)
        weather_combo.grid(row=5, column=1, padx=5, pady=5)
        
        # Predict button
        predict_btn = tk.Button(input_frame, text="Predict Delay", command=self.predict_flight,
                               bg="#4CAF50", fg="white", font=("Arial", 12, "bold"))
        predict_btn.grid(row=6, column=0, columnspan=2, pady=15)
        
        # Right frame - Dataset display
        right_frame = tk.Frame(main_frame)
        right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)
        
        # Dataset frame
        dataset_frame = tk.LabelFrame(right_frame, text="All Flight Records (500 records)", 
                                    font=("Arial", 12, "bold"))
        dataset_frame.pack(fill=tk.BOTH, expand=True)
        
        # Treeview for displaying data
        columns = ("Airline", "Source", "Dest", "Distance", "Hour", "Weather", "Delay")
        self.tree = ttk.Treeview(dataset_frame, columns=columns, show="headings", height=20)
        
        # Define headings
        for col in columns:
            self.tree.heading(col, text=col)
            self.tree.column(col, width=100)
        
        # Scrollbar for treeview
        scrollbar = ttk.Scrollbar(dataset_frame, orient=tk.VERTICAL, command=self.tree.yview)
        self.tree.configure(yscrollcommand=scrollbar.set)
        
        # Pack treeview and scrollbar
        self.tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # Load data button
        load_btn = tk.Button(dataset_frame, text="Refresh Data", command=self.load_data,
                           bg="#2196F3", fg="white")
        load_btn.pack(pady=5)
        
        # Status label
        self.status_var = tk.StringVar(value="Ready")
        status_label = tk.Label(right_frame, textvariable=self.status_var, 
                              relief=tk.SUNKEN, anchor=tk.W)
        status_label.pack(side=tk.BOTTOM, fill=tk.X)
        
        # Load initial data
        self.load_data()
    
    def load_data(self):
        # Clear existing data
        for item in self.tree.get_children():
            self.tree.delete(item)
        
        # Load all records
        original_df = pd.read_csv("flights.csv")
        for index, row in original_df.iterrows():
            delay_status = "Delay" if row['delay'] == 1 else "On Time"
            self.tree.insert("", tk.END, values=(
                row['airline'], row['source'], row['destination'],
                row['distance'], row['departure_hour'], row['weather'], delay_status
            ))
        self.status_var.set(f"Loaded {len(original_df)} records")
    
    def predict_flight(self):
        try:
            airline = self.airline_var.get()
            source = self.source_var.get()
            destination = self.dest_var.get()
            distance = int(self.distance_var.get())
            hour = int(self.hour_var.get())
            weather = self.weather_var.get()
            
            prediction = predict_flight(airline, source, destination, distance, hour, weather)
            
            # Show prediction in messagebox and status
            messagebox.showinfo("Prediction Result", 
                              f"Flight Status: **{prediction}**\n\n"
                              f"Details:\n"
                              f"Airline: {airline}\n"
                              f"Source: {source}\n"
                              f"Destination: {destination}\n"
                              f"Distance: {distance} km\n"
                              f"Departure: {hour}:00\n"
                              f"Weather: {weather}")
            
            self.status_var.set(f"Predicted: {prediction} for {airline} {source}-{destination}")
            
        except Exception as e:
            messagebox.showerror("Error", f"Prediction failed: {str(e)}")

def main():
    root = tk.Tk()
    app = FlightPredictorApp(root)
    root.mainloop()

if __name__ == "__main__":
    main()

Dataset created 
ML Accuracy: 1.0
Epoch: 0 Loss: 0.6579394340515137
Epoch: 10 Loss: 0.6149449944496155
Epoch: 20 Loss: 0.571908175945282
Epoch: 30 Loss: 0.5263713002204895
Epoch: 40 Loss: 0.4774698317050934
DL Accuracy: 0.8
